In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

aerial_cactus_identification_path = kagglehub.competition_download('aerial-cactus-identification')

print('Data source import complete.')


# 시드값 고정 및 GPU 장비 설정

## 시드값 고정

In [ ]:
import torch
import random
import numpy as np
import os

In [ ]:
seed = 50 # 시드값 고정
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)                         # 파이썬 난수 생성기 시드 고정
np.random.seed(seed)                      # 넘파이 난수 생성기 시드 고정
torch.manual_seed(seed)                   # 파이토치 난수 생성기 시드 고정 (CPU 사용 시)
torch.cuda.manual_seed(seed)              # 파이토치 난수 생성기 시드 고정 (GPU 사용 시)
torch.cuda.manual_seed_all(seed)          # 파이토치 난수 생성기 시드 고정 (멀티 GPU 사용 시)
torch.backends.cudnn.deterministic = True # 확정적 연산 사용
torch.backends.cudnn.beenchmark = False   # 벤치마크 기능 해제
torch.backends.cudnn.enabled = False      # cudnn 사용 해제

## GPU 장비 설정

In [ ]:
# if torch.cuda.is_available():
#     device = torch.device('cuda')
# else:
#     device = torch.device('cpu')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

# 데이터 준비

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data_path = '/kaggle/input/aerial-cactus-identification/'

In [ ]:
labels = pd.read_csv(data_path + 'train.csv')
submission = pd.read_csv(data_path + 'sample_submission.csv')

In [ ]:
from zipfile import ZipFile

In [ ]:
with ZipFile(data_path + 'train.zip') as zipper:
    zipper.extractall()

In [ ]:
with ZipFile(data_path + 'test.zip') as zipper:
    zipper.extractall()

## 훈련 데이터, 검증 데이터 분리

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train, valid = train_test_split(labels,
                                test_size=0.1,
                                stratify=labels['has_cactus'],
                                random_state=50)

In [ ]:
print('훈련 데이터 개수: ', len(train))
print('검증 데이터 개수: ', len(valid))

## 데이터셋 클래스 정의

In [ ]:
import cv2
from torch.utils.data import Dataset

In [ ]:
class ImageDataset(Dataset):
    # 초기화 메서드(생성자)
    def __init__(self, df, img_dir='./', transform=None):
        super().__init__()
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    # 데이터셋 크기 반환 메서드
    def __len__(self):
        return len(self.df)

    # 인덱스(idx)에 해당하는 데이터 반환 메서드
    def __getitem__(self, idx):
        img_id = self.df.iloc[idx, 0]                  # 이미지 ID
        img_path = self.img_dir + img_id               # 이미지 파일 경로
        image = cv2.imread(img_path)                   # 이미지 파일 읽기
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # 이미지 색상 보정
        label = self.df.iloc[idx, 1]                   # 이미지 레이블(타깃값)

        if self.transform is not None:
            image = self.transform(image)              # 변환기가 있다면 이미지 변환

        return image, label

## 데이터셋 생성

In [ ]:
from torchvision import transforms

In [ ]:
transform = transforms.ToTensor()

In [ ]:
dataset_train = ImageDataset(df=train, img_dir='train/', transform=transform)
dataset_valid = ImageDataset(df=valid, img_dir='train/', transform=transform)

## 데이터 로더 생성

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
loader_train = DataLoader(dataset=dataset_train, batch_size=32, shuffle=True)
loader_valid = DataLoader(dataset=dataset_valid, batch_size=32, shuffle=False)

# 모델 생성

In [ ]:
import torch.nn as nn
import torch.nn.functional as F # 신경망 모듈에서 자주 사용되는 함수

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        # 첫 번째 합성곱 계층
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=2)

        # 두 번째 합성곱 계층
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=2)

        # 최대 풀링 계층
        self.max_pool = nn.MaxPool2d(kernel_size=2)
        # 평균 풀링 계층
        self.avg_pool = nn.AvgPool2d(kernel_size=2)
        # 전결합 계층
        self.fc = nn.Linear(in_features=64 * 4 * 4, out_features=2)

    # 순전파 출력 정의
    def forward(self, x):
        x = self.max_pool(F.relu(self.conv1(x)))
        x = self.max_pool(F.relu(self.conv2(x)))
        x = self.avg_pool(x)
        x = x.view(-1, 64 * 4 * 4) # 평탄화
        x = self.fc(x)
        return x

In [ ]:
model = Model().to(device)

In [ ]:
# class Model(nn.Module):
#     def __init__(self):
#         super().__init__()

#         # 첫 번째 합성곱 계층
#         self.layer1 = nn.Sequential(nn.Conv2d(in_channels=3,
#                                               out_channels=32,
#                                               kernel_size=3,
#                                               padding=2),
#                                    nn.ReLU(),
#                                    nn.MaxPool2d(kernel_size=2))

#         # 두 번째 합성곱 계층
#         self.layer2 = nn.Sequential(nn.Conv2d(in_channels=32,
#                                               out_channels=64,
#                                               kernel_size=3,
#                                               padding=2),
#                                    nn.ReLU(),
#                                    nn.MaxPool2d(kernel_size=2))

#         # 평균 풀링 계층
#         self.avg_pool = nn.AvgPool2d(kernel_size=2)
#         # 전결합 계층
#         self.fc = nn.Linear(in_features=64 * 4 * 4, out_features=2)

#     # 순전파 출력 정의
#     def forward(self, x):
#         x = self.layer1(F.relu(self.conv1(x)))
#         x = self.layer2(F.relu(self.conv2(x)))
#         x = self.avg_pool(x)
#         x = x.view(-1, 64 * 4 * 4) # 평탄화
#         x = self.fc(x)
#         return x

# 모델 훈련

## 손실 함수 설정

In [ ]:
criterion = nn.CrossEntropyLoss()

## 옵티마이저 설정

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

## 모델 훈련

In [ ]:
epochs = 10

In [ ]:
for epoch in range(epochs):
    epoch_loss = 0 # 에폭 별 손실값 초기화

    # '반복 횟수'만큼 반복
    for images, labels in loader_train:
        # 이미지, 레이블 데이터 미니배치를 장비에 할당
        images = images.to(device)
        labels = labels.to(device)

        # 옵티마이저 내 기울기 초기화
        optimizer.zero_grad()
        # 순전파: 이미지 데이터를 신경망 모델의 입력값으로 사용해 출력값 계산
        outputs = model(images)
        # 손실 함수를 활용해 outputs와 labels의 손실값 계산
        loss = criterion(outputs, labels)
        # 현재 배치에서의 손실 추가
        epoch_loss += loss.item()
        loss.backward()
        # 가중치 갱신
        optimizer.step()

    # 훈련 데이터 손실값 출력
    print(f'에폭 [{epoch+1}/{epochs}] - 손실값: {epoch_loss/len(loader_train):.4f}')

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
true_list = []
preds_list = []

In [ ]:
model.eval() # 모델을 평가 상태로 설정

In [ ]:
with torch.no_grad(): # 기울기 계싼 비활성화
    for images, labels in loader_valid:
        # 이미지, 레이블 데이터 미니배치를 장비에 할당
        images = images.to(device)
        labels = labels.to(device)

        # 순전파: 이미지 데이터를 신경망 모델의 입력값으로 사용해 출력값 계산
        outputs = model(images)
        preds = torch.softmax(outputs.cpu(), dim=1)[:, 1] # 예측 확률
        true = labels.cpu() # 실제값
        # 예측 확률과 실젯값을 리스트에 추가
        preds_list.extend(preds)
        true_list.extend(true)

In [ ]:
# 검증 데이터 ROC AUC 점수 계산
print(f'검증 데이터 ROC AUC: {roc_auc_score(true_list, preds_list):.4f}')

# 예측 및 결과 제출

In [ ]:
dataset_test = ImageDataset(df=submission, img_dir='test/', transform=transform)
loader_test = DataLoader(dataset=dataset_test, batch_size=32, shuffle=False)

## 예측

In [ ]:
model.eval()

In [ ]:
preds = []

In [ ]:
with torch.no_grad(): # 기울기 계산 비활성화
    for images, _ in loader_test:
        # 이미지 데이터 미니배치를 장비에 할당
        images = images.to(device)

        # 순전파
        outputs = model(images)
        # 타깃값이 1일 확률(예측값)
        preds_part = torch.softmax(outputs.cpu(), dim=1)[:, 1].tolist()
        # preds에 preds_part 이어붙기기
        preds.extend(preds_part)

## 결과 제출

In [ ]:
submission['has_cactus'] = preds
submission.to_csv('submission.csv', index=False)

## 압축이 풀린 이미지 삭제

In [ ]:
import shutil

In [ ]:
shutil.rmtree('./train')
shutil.rmtree('./test')

# 성능 개선

In [ ]:
import torch
import random
import numpy as np
import os

In [ ]:
seed = 50 # 시드값 고정
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)                         # 파이썬 난수 생성기 시드 고정
np.random.seed(seed)                      # 넘파이 난수 생성기 시드 고정
torch.manual_seed(seed)                   # 파이토치 난수 생성기 시드 고정 (CPU 사용 시)
torch.cuda.manual_seed(seed)              # 파이토치 난수 생성기 시드 고정 (GPU 사용 시)
torch.cuda.manual_seed_all(seed)          # 파이토치 난수 생성기 시드 고정 (멀티 GPU 사용 시)
torch.backends.cudnn.deterministic = True # 확정적 연산 사용
torch.backends.cudnn.beenchmark = False   # 벤치마크 기능 해제
torch.backends.cudnn.enabled = False      # cudnn 사용 해제

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 데이터 준비

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data_path = '/kaggle/input/aerial-cactus-identification/'

In [ ]:
labels = pd.read_csv(data_path + 'train.csv')
submission = pd.read_csv(data_path + 'sample_submission.csv')

In [ ]:
from zipfile import ZipFile

In [ ]:
with ZipFile(data_path + 'train.zip') as zipper:
    zipper.extractall()

In [ ]:
with ZipFile(data_path + 'test.zip') as zipper:
    zipper.extractall()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train, valid = train_test_split(labels,
                                test_size=0.1,
                                stratify=labels['has_cactus'],
                                random_state=50)

In [ ]:
print('훈련 데이터 개수: ', len(train))
print('검증 데이터 개수: ', len(valid))

In [ ]:
import cv2
from torch.utils.data import Dataset

In [ ]:
class ImageDataset(Dataset):
    # 초기화 메서드(생성자)
    def __init__(self, df, img_dir='./', transform=None):
        super().__init__()
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    # 데이터셋 크기 반환 메서드
    def __len__(self):
        return len(self.df)

    # 인덱스(idx)에 해당하는 데이터 반환 메서드
    def __getitem__(self, idx):
        img_id = self.df.iloc[idx, 0]                  # 이미지 ID
        img_path = self.img_dir + img_id               # 이미지 파일 경로
        image = cv2.imread(img_path)                   # 이미지 파일 읽기
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # 이미지 색상 보정
        label = self.df.iloc[idx, 1]                   # 이미지 레이블(타깃값)

        if self.transform is not None:
            image = self.transform(image)              # 변환기가 있다면 이미지 변환

        return image, label

### 이미지 변환기 정의

In [ ]:
from torchvision import transforms

In [ ]:
# 훈련 데이터용 변환기
transform_train = transforms.Compose([transforms.ToTensor(),
                                      transforms.Pad(32, padding_mode='symmetric'),
                                      transforms.RandomHorizontalFlip(),
                                      transforms.RandomVerticalFlip(),
                                      transforms.RandomRotation(10),
                                      transforms.Normalize((0.485, 0.456, 0.406),
                                                           (0.229, 0.224, 0.225))])

In [ ]:
# 검증 및 테스트 데이터용 변환기
transform_test = transforms.Compose([transforms.ToTensor(),
                                     transforms.Pad(32, padding_mode='symmetric'),
                                     transforms.Normalize((0.485, 0.456, 0.406),
                                                           (0.229, 0.224, 0.225))])

### 데이터셋 및 데이터 로더 생성

In [ ]:
dataset_train = ImageDataset(df=train, img_dir='train/',
                             transform=transform_train)

In [ ]:
dataset_valid = ImageDataset(df=valid, img_dir='train/',
                             transform=transform_test)

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
loader_train = DataLoader(dataset=dataset_train, batch_size=32, shuffle=True)
loader_valid = DataLoader(dataset=dataset_valid, batch_size=32, shuffle=False)

## 모델 생성

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        # 1 ~ 5 번째 (합성곱, 배치 정규화, 최대 풀링) 계층
        self.layer1 = nn.Sequential(nn.Conv2d(in_channels=3,
                                              out_channels=32,
                                              kernel_size=3,
                                              padding=2),
                                    nn.BatchNorm2d(32),  # 배치 정규화
                                    nn.LeakyReLU(),      # LeakyReLU 활성화 함수
                                    nn.MaxPool2d(kernel_size=2))

        self.layer2 = nn.Sequential(nn.Conv2d(in_channels=32,
                                              out_channels=64,
                                              kernel_size=3,
                                              padding=2),
                                    nn.BatchNorm2d(64),
                                    nn.LeakyReLU(),
                                    nn.MaxPool2d(kernel_size=2))

        self.layer3 = nn.Sequential(nn.Conv2d(in_channels=64,
                                              out_channels=128,
                                              kernel_size=3,
                                              padding=2),
                                    nn.BatchNorm2d(128),
                                    nn.LeakyReLU(),
                                    nn.MaxPool2d(kernel_size=2))

        self.layer4 = nn.Sequential(nn.Conv2d(in_channels=128,
                                              out_channels=256,
                                              kernel_size=3,
                                              padding=2),
                                    nn.BatchNorm2d(256),
                                    nn.LeakyReLU(),
                                    nn.MaxPool2d(kernel_size=2))

        self.layer5 = nn.Sequential(nn.Conv2d(in_channels=256,
                                              out_channels=512,
                                              kernel_size=3,
                                              padding=2),
                                    nn.BatchNorm2d(512),
                                    nn.LeakyReLU(),
                                    nn.MaxPool2d(kernel_size=2))

        # 평균 풀링 계층
        self.avg_pool = nn.AvgPool2d(kernel_size=4)

        # 전결합 계층
        self.fc1 = nn.Linear(in_features=512 * 1 * 1, out_features=64)
        self.fc2 = nn.Linear(in_features=64, out_features=2)

    # 순전파 출력 정의
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.avg_pool(x)
        x = x.view(-1, 512 * 1 * 1) # 평탄화
        x = self.fc1(x)
        x = self.fc2(x)
        return x

In [ ]:
model = Model().to(device)

## 모델 훈련

In [ ]:
# 손실 함수
criterion = nn.CrossEntropyLoss()

In [ ]:
# 옵티마이저
optimizer = torch.optim.Adamax(model.parameters(), lr=0.00006)

In [ ]:
epochs = 70

In [ ]:
for epoch in range(epochs):
    epoch_loss = 0

    for images, labels in loader_train:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        epoch_loss += loss.item()
        loss.backward()
        optimizer.step()

    print(f'에폭 [{epoch+1}/{epochs}] - 손실값: {epoch_loss/len(loader_train):.4f}')

## 성능 검증

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
true_list = []
preds_list = []

In [ ]:
model.eval()

In [ ]:
with torch.no_grad(): # 기울기 계산 비활성화
    for images, labels in loader_valid:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = torch.softmax(outputs.cpu(), dim=1)[:, 1]
        true = labels.cpu()
        preds_list.extend(preds)
        true_list.extend(true)

In [ ]:
print(f'검증 데이터 ROC AUC: {roc_auc_score(true_list, preds_list):.4f}')

## 예측 및 결과 제출

In [ ]:
dataset_test = ImageDataset(df=submission, img_dir='test/', transform=transform_test)
loader_test = DataLoader(dataset=dataset_test, batch_size=32, shuffle=False)

In [ ]:
model.eval()

In [ ]:
preds = []

In [ ]:
with torch.no_grad(): # 기울기 계산 비활성화
    for images, _ in loader_test:
        images = images.to(device)

        outputs = model(images)
        preds_part = torch.softmax(outputs.cpu(), dim=1)[:, 1].tolist()
        preds.extend(preds_part)

In [ ]:
submission['has_cactus'] = preds
submission.to_csv('submission.csv', index=False)

In [ ]:
import shutil

In [ ]:
shutil.rmtree('./train')
shutil.rmtree('./test')